In [ ]:
first_name = "Gregory"
last_name = "Miller"
email = "millegre001@tamu.edu"

In [1]:
import numpy as np
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import itertools
from scipy.optimize import linprog
import pickle
import random
import joblib
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import defaultdict
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.utils import resample, class_weight
from collections import OrderedDict
from matplotlib.ticker import MaxNLocator

2025-04-24 13:51:00.341451: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-24 13:51:00.503078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745520660.596868   13946 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745520660.622042   13946 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-24 13:51:00.786252: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [ ]:
#The function below will create the G matrix given the parameters n and k where it creates a k size identity matrix 
#and appends a k x n-k -100 to 100 random real numbers

In [ ]:
def G_generator(n, k):
    G = np.identity(k)
    matrix = np.random.uniform(-100, 100, (k, n - k)).astype(np.float32)
    G = np.concatenate((G, matrix), axis=1)
    #print(G)
    return G

In [ ]:
#The function below will make the shape to the max size and flatten it.

In [ ]:
def pad_G(G, max_n = 10, max_k = 6):
    k,n = G.shape
    padded = np.zeros((max_k, max_n), dtype=G.dtype)
    padded[:k,:n] = G
    return padded.flatten()

In [ ]:
#The function below is the tuple generator. 

In [ ]:
def Tuple_generator(n, m):
    tuples = []
    elements = list(range(n))
    for a in elements:
        for b in elements:
            if a != b:
                remaining = [x for x in elements if x not in {a,b}]
                X_sets = itertools.combinations(remaining, m-1) if m - 1 > 0 else [()]
                for X in X_sets:
                    psi_values = itertools.product([-1, 1], repeat = m)
                    for psi in psi_values:
                        tuples.append((a,b,X,psi))
    return tuples

In [ ]:
#The below follows the Linear progression file shown in the project pdf

In [ ]:
def LP_solver(G, a, b, X, psi):
    k, n = G.shape
    bounds = [(None, None)] * k

    X_sorted = sorted(X)
    Y = [x for x in range(n) if x not in {a,b} and x not in X_sorted]
    Y_sorted = sorted(Y)
    x_values = [a] + X_sorted + [b] + Y_sorted
    tau_inv = {val: i for i, val in enumerate(x_values)}
    #negated for minimization as used by the scipy.linprog
    c = [(-psi[0] * G[i, a]) for i in range(k)]
    c = np.array(c)

    A_ub = []
    b_ub = []
    
    for j in X_sorted:
        row = [(psi[tau_inv[j]] * G[i,j] - psi[0] * G[i, a]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(0)

    for j in X_sorted:
        row = [(-psi[tau_inv[j]] * G[i, j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(-1)

    A_eq = [(G[i,b]) for i in range(k)]
    b_eq = [1]

    for j in Y_sorted:
        row = [(G[i, j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(1)

    for j in Y_sorted:
        row = [(-G[i,j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(1)

    
    A_ub = np.array(A_ub)
    b_ub = np.array(b_ub)
    A_eq = np.array(A_eq).reshape(1,k)
    b_eq = np.array(b_eq)
    #print("shape of c ", c.shape)
    #print("shape of A_ub ", A_ub.shape)
    #print("shape of b_ub ", b_ub.shape)
    #print("shape of A_eq ", A_eq.shape)
    #print("shape of b_eq ", b_eq.shape)
    
    res = linprog(c, A_ub = A_ub, b_ub = b_ub, A_eq = A_eq, b_eq = b_eq, bounds = bounds, method = 'highs')

    #print(f"Tuple: a={a}, b={b}, X={X}, psi={psi}")
    #print(f"Objective value (raw): {-res.fun if res.success else 'N/A'}")
    #print(f"Solver success: {res.success}, status: {res.status}")
    if res.success:
        return -res.fun
    elif res.status == 3:
        return float("inf")
    else:
        return 0

In [ ]:
#The below function calls all of the above functions and will return the max h_m or inf

In [ ]:
def H_m_computer(G, n, k):
    tuples = Tuple_generator(n,k)
    h_m = max(LP_solver(G, *tpl) for tpl in tuples)
    if h_m == float("inf"):
        return float("inf")
    return h_m


In [ ]:
#The below code below will create 500 samples of the specified n, k, and m value

In [ ]:
def generate_dataset(n,k,m,num_samples_per_config=500):
    config_dataset = []
    num_inf = 0
    max_inf = 0.2 * m * num_samples_per_config
    while len(config_dataset) < num_samples_per_config:
        G = G_generator(n,k)
        h_m = H_m_computer(G,n,m)
        padded_G = pad_G(G)
        sample_data = {'n': n, 'k': k, 'm': m, 'G': padded_G, 'h_m': h_m}
        if h_m == float("inf"):
            if num_inf < max_inf:
                config_dataset.append(sample_data)
                num_inf += 1
            else:
                continue
        else:
            config_dataset.append(sample_data)
    return config_dataset

In [ ]:
#split the below up so I can run them one after another even if the system stops
# I am running Jupyter Notebook in WSL which is limited to only 1.6GB, because of this I originally had it run and append to one large dataset.
#since that took over 20 hours and was only halfway I broke up dataset generation and saving into the 21 different combinations of n,k, and m.
#Then I combined and randomized the data

In [ ]:
dataset = []
dataset = generate_dataset(9,4,2)
with open('dataset2_1.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,3)
with open('dataset2_2.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,4)
with open('dataset2_3.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,5)
with open('dataset2_4.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,2)
with open('dataset2_5.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,3)
with open('dataset2_6.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,4)
with open('dataset2_7.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,6,2)
with open('dataset2_8.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,6,3)
with open('dataset2_9.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,2)
with open('dataset2_10.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,3)
with open('dataset2_11.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,4)
with open('dataset2_12.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,5)
with open('dataset2_13.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,6)
with open('dataset2_14.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,2)
with open('dataset2_15.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,3)
with open('dataset2_16.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,4)
with open('dataset2_17.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,5)
with open('dataset2_18.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,2)
with open('dataset2_19.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,3)
with open('dataset2_20.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,4)
with open('dataset2_21.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
#I stored my Gs that I created from my code and switch them to only be Ps and flatten them

In [ ]:
def transform_G(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['G'],(6,10))
    P = G_mat[:,k:]
    padded = np.zeros((6,6), dtype=P.dtype)
    padded[:P.shape[0], :P.shape[1]] = P

    return padded.flatten()

In [ ]:
#From the data that I am using from the dataset that keegan created I need to reformat P to fit my model

In [ ]:
def transform_P(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['P'],(k,n-k))
    padded = np.zeros((6,6), dtype=G_mat.dtype)
    padded[:G_mat.shape[0], :G_mat.shape[1]] = G_mat

    return padded.flatten()
    

In [ ]:
#Below loads all of my datasets and splits each before combining them into the train val test sections
#This means that for each n,k,m have the same amount in each train val and test

In [ ]:
files = [
    "dataset1.pkl",
    "dataset2.pkl",
    "dataset3.pkl",
    "dataset4.pkl",
    "dataset5.pkl",
    "dataset6.pkl",
    "dataset7.pkl",
    "dataset8.pkl",
    "dataset9.pkl",
    "dataset10.pkl",
    "dataset11.pkl",
    "dataset12.pkl",
    "dataset13.pkl",
    "dataset14.pkl",
    "dataset15.pkl",
    "dataset16.pkl",
    "dataset17.pkl",
    "dataset18.pkl",
    "dataset19.pkl",
    "dataset20.pkl",
    "dataset21.pkl",
    "dataset2_1.pkl",
    "dataset2_2.pkl",
    "dataset2_3.pkl",
    "dataset2_4.pkl",
    "dataset2_5.pkl",
    "dataset2_6.pkl",
    "dataset2_7.pkl",
    "dataset2_8.pkl",
    "dataset2_9.pkl",
    "dataset2_10.pkl",
    "dataset2_11.pkl",
    "dataset2_12.pkl",
    "dataset2_13.pkl",
    "dataset2_14.pkl",
    "dataset2_15.pkl",
    "dataset2_16.pkl",
    "dataset2_17.pkl",
    "dataset2_18.pkl",
    "dataset2_19.pkl",
    "dataset2_20.pkl",
    "dataset2_21.pkl",
]

train_nkm, val_nkm, test_nkm = [],[],[]
train_G, val_G, test_G = [],[],[]
train_y, val_y, test_y = [],[],[]
for file in files:
    X_params = []
    X_G = []
    y = []
    with open(file, "rb") as f:
        data = joblib.load(f)
    for sample in data:
        X_params.append((sample['n'],sample['k'],sample['m']))
        X_G.append(transform_G(sample))
        y.append(sample['h_m'])
    nkm = np.array(X_params, dtype=np.float32)
    G = np.array(X_G, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    part_train_nkm, temp_nkm, part_train_G, temp_G, part_train_y, temp_y = train_test_split(nkm, G, y, test_size=0.3)
    part_val_nkm, part_test_nkm, part_val_G, part_test_G, part_val_y, part_test_y = train_test_split(temp_nkm, temp_G, temp_y, test_size=0.33)
    train_nkm.append(part_train_nkm)
    train_G.append(part_train_G)
    train_y.append(part_train_y)
    val_nkm.append(part_val_nkm)
    val_G.append(part_val_G)
    val_y.append(part_val_y)
    test_nkm.append(part_test_nkm)
    test_G.append(part_test_G)
    test_y.append(part_test_y)


#for sample in data_records:
#    X_params.append((sample['n'],sample['k'],sample['m']))
#    X_G.append(transform_P(sample))
#    y.append(sample['result'])

In [ ]:
#loads keegans set and puts it into a dictionary

In [ ]:
other_data = joblib.load("results_dataframe.pkl")
print(len(other_data))
data_records = other_data.to_dict('records')
print(len(data_records))

In [ ]:
#splits the data by the nkm

In [ ]:
data_by_config = defaultdict(list)
for sample in data_records:
    key = (sample['n'], sample['k'], sample['m'])
    data_by_config[key].append(sample)

In [ ]:
#stores the split data into a bunch of pkl files bc I was crashing whenever I loaded too much at once

In [ ]:
for key, samples in data_by_config.items():
    n, k, m = key
    filename = f"dataset_n{n}_k{k}_m{m}.pkl"
    joblib.dump(samples, filename)

In [ ]:
#below loads the datasets from keegan and takes 10000 values of each and splits these 10k into train val and test
#again evenly split

In [ ]:
files = [
    "dataset_n9_k4_m2.pkl",
    "dataset_n9_k4_m3.pkl",
    "dataset_n9_k4_m4.pkl",
    "dataset_n9_k4_m5.pkl",
    "dataset_n9_k5_m2.pkl",
    "dataset_n9_k5_m3.pkl",
    "dataset_n9_k5_m4.pkl",
    "dataset_n9_k6_m2.pkl",
    "dataset_n9_k6_m3.pkl",
    "dataset_n10_k4_m2.pkl",
    "dataset_n10_k4_m3.pkl",
    "dataset_n10_k4_m4.pkl",
    "dataset_n10_k4_m5.pkl",
    "dataset_n10_k4_m6.pkl",
    "dataset_n10_k5_m2.pkl",
    "dataset_n10_k5_m3.pkl",
    "dataset_n10_k5_m4.pkl",
    "dataset_n10_k5_m5.pkl",
    "dataset_n10_k6_m2.pkl",
    "dataset_n10_k6_m3.pkl",
    "dataset_n10_k6_m4.pkl"
]

for file in files:
    X_params = []
    X_G = []
    y = []
    with open(file, "rb") as f:
        data = joblib.load(f)
    for sample in data[:10000]:
        X_params.append((sample['n'],sample['k'],sample['m']))
        X_G.append(transform_P(sample))
        y.append(sample['result'])
    nkm = np.array(X_params, dtype=np.float32)
    G = np.array(X_G, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    part_train_nkm, temp_nkm, part_train_G, temp_G, part_train_y, temp_y = train_test_split(nkm, G, y, test_size=0.3)
    part_val_nkm, part_test_nkm, part_val_G, part_test_G, part_val_y, part_test_y = train_test_split(temp_nkm, temp_G, temp_y, test_size=0.33)
    train_nkm.append(part_train_nkm)
    train_G.append(part_train_G)
    train_y.append(part_train_y)
    val_nkm.append(part_val_nkm)
    val_G.append(part_val_G)
    val_y.append(part_val_y)
    test_nkm.append(part_test_nkm)
    test_G.append(part_test_G)
    test_y.append(part_test_y)

train_nkm = np.vstack(train_nkm)
train_G = np.vstack(train_G)
train_y = np.concatenate(train_y)
val_nkm = np.vstack(val_nkm)
val_G = np.vstack(val_G)
val_y = np.concatenate(val_y)
test_nkm = np.vstack(test_nkm)
test_G = np.vstack(test_G)
test_y = np.concatenate(test_y)

print(train_nkm.shape,val_nkm.shape,test_nkm.shape)

In [ ]:
#loading the saved 221k data points that I sorted
# I have taken the nkm and P and made them into some variations of those as well as G

In [ ]:
nkm_list = [
    (9,4,2),(9,4,3),(9,4,4),(9,4,5),
    (9,5,2),(9,5,3),(9,5,4),
    (9,6,2),(9,6,3),
    (10,4,2),(10,4,3),(10,4,4),(10,4,5),(10,4,6),
    (10,5,2),(10,5,3),(10,5,4),(10,5,5),
    (10,6,2),(10,6,3),(10,6,4),
]
splits = {}
for n,k,m in nkm_list:
    key = f"{n}_{k}_{m}"
    
    nkm = joblib.load(f"params_n{float(n)}_k{float(k)}_m{float(m)}.pkl")
    G   = joblib.load(f"G_n{float(n)}_k{float(k)}_m{float(m)}.pkl")
    P = []
    for i in range(len(G)):
        P.append(G[i].reshape(6,10))
    P = np.array(P, dtype=np.float32)
    y   = joblib.load(f"y_n{float(n)}_k{float(k)}_m{float(m)}.pkl")
    y_log = np.log2(y).reshape(-1, 1)
    splits[key] = {
        "nkm": nkm,
        "G": P,
        "y_log": y_log
    }

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming you have a dict `splits` where for each key "n_k_m"
# splits[key]["test_y_log"] is your 1‑D array of log2(h_m) values.

configs = list(splits.keys())          # e.g. ["9_4_2","9_4_3",…]
data = [ np.array(splits[c]["y_log"]).flatten() for c in configs ]

plt.figure(figsize=(14,6))
plt.boxplot(data, labels=configs, showfliers=False)
plt.xticks(rotation=90)
plt.title("Distribution of log₂(hₘ) by (n,k,m)")
plt.xlabel("(n,k,m)")
plt.ylabel("log₂(hₘ)")
plt.tight_layout()
plt.show()

In [ ]:
import math

configs = list(splits.keys())
n = len(configs)
cols = 6
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(24, 16), sharey=False)
axes = axes.flatten()

for ax, cfg in zip(axes, configs):
    y = np.array(splits[cfg]['y_log']).flatten()
    ax.hist(y, bins=40)
    ax.set_title(cfg, fontsize=10)
    ax.set_xlabel('log₂(hₘ)', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.tick_params(axis='both', which='major', labelsize=6)

# Hide any unused subplots
for ax in axes[n:]:
    ax.axis('off')

plt.suptitle("Histograms of log₂(hₘ) by (n,k,m)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
subset = ["9_4_5", "9_5_4", "9_6_3", "10_4_6", "10_5_5", "10_6_4"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)
axes = axes.flatten()

for ax, cfg in zip(axes, subset):
    # Retrieve your log2(h_m) data for this configuration
    y = np.array(splits[cfg]['y_log']).flatten()
    ax.hist(y, bins=30)
    
    # set title and labels
    ax.set_title(f"{cfg.replace('_', ', ')}", fontsize=12)
    ax.set_xlabel('log₂(hₘ)', fontsize=15)
    ax.set_ylabel('Count', fontsize=10)
    ax.tick_params(axis='both', labelsize=15)
    
    # force more x‑ticks: use MaxNLocator for ~8 ticks
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8, prune=None))
    # optionally, rotate them slightly for readability
    for label in ax.get_xticklabels():
        label.set_rotation(30)

plt.suptitle("Histograms of log₂(hₘ) for Selected (n,k,m) Configs", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
def kmeans(train_y_log, n_clusters=1):
    km = KMeans(n_clusters=n_clusters).fit(train_y_log)
    y_binned = km.labels_
    centers = km.cluster_centers_.flatten()
    order = np.argsort(centers)
    label_map = {orig:i for i,orig in enumerate(order)}
    y_train_bins = np.vectorize(label_map.get)(y_binned)
    return y_train_bins

In [ ]:
for key, data in splits.items():
    clusters = 1
    # if key in subset:
    #     clusters = 5
    n, k, m = map(int, key.split("_"))
    y_bins = kmeans(data['y_log'])
    data['y_bins'] = y_bins
    joblib.dump(y_bins, f"y_bins_n{float(n)}_k{float(k)}_m{float(m)}.pkl")

In [ ]:
for key, data in splits.items():
    Xnkm   = np.asarray(data["nkm"])
    XG     = np.asarray(data["G"])
    ylog   = data["y_log"].reshape(-1,1)
    bins   = data['y_bins']

    nkm_tr, nkm_te, G_tr, G_te, ylog_tr, ylog_te, bins_tr, bins_te = train_test_split(
        Xnkm, XG, ylog, bins,
        test_size=0.1,
        shuffle=True)

    base = key
    joblib.dump(nkm_tr,   f"train_nkm_{base}.pkl")
    joblib.dump(G_tr,     f"train_G_{  base}.pkl")
    joblib.dump(ylog_tr,  f"train_ylog_{base}.pkl")
    joblib.dump(bins_tr, f"train_bins_{base}.pkl")

    joblib.dump(nkm_te,   f"test_nkm_{base}.pkl")
    joblib.dump(G_te,     f"test_G_{  base}.pkl")
    joblib.dump(ylog_te,  f"test_ylog_{base}.pkl")
    joblib.dump(bins_te, f"test_bins_{base}.pkl")

    print(f"{base}: train {nkm_tr.shape[0]}, test {nkm_te.shape[0]}")

In [2]:
splits = {}
nkm_list = [
    (9,4,2),(9,4,3),(9,4,4),(9,4,5),
    (9,5,2),(9,5,3),(9,5,4),
    (9,6,2),(9,6,3),
    (10,4,2),(10,4,3),(10,4,4),(10,4,5),(10,4,6),
    (10,5,2),(10,5,3),(10,5,4),(10,5,5),
    (10,6,2),(10,6,3),(10,6,4),
]
for n,k,m in nkm_list:
    key = f"{n}_{k}_{m}"
    nkm = joblib.load(f"train_nkm_{n}_{k}_{m}.pkl")
    G   = joblib.load(f"train_G_{n}_{k}_{m}.pkl")
    y   = joblib.load(f"train_ylog_{n}_{k}_{m}.pkl")
    bins= joblib.load(f"train_bins_{n}_{k}_{m}.pkl")
    splits[key] = {
        "nkm":nkm,
        "G":G,
        "y_bins":bins,
        "y_log":y
    }

In [3]:
def balance_split(Xnkm, XG, bins, ylog):
    unique_bins, counts = np.unique(bins, return_counts=True)
    max_count = max(counts)
    Xnkm_bal, XG_bal, bins_bal, y_bal = [], [], [], []
    for b, cnt in zip(unique_bins, counts):
        mask = (bins == b)
        Xnkm_b = Xnkm[mask]
        XG_b = XG[mask]
        bins_b = bins[mask]
        yb = ylog[mask]
        Xnkm_up, XG_up, bins_up, yb_up = resample(
        Xnkm_b, XG_b, bins_b, yb,
        replace=True, n_samples=max_count)
        Xnkm_bal.append(Xnkm_up)
        XG_bal.append(XG_up)
        bins_bal.append(bins_up)
        y_bal.append(yb_up)
    return (np.vstack(Xnkm_bal), np.vstack(XG_bal), np.concatenate(bins_bal), np.concatenate(y_bal))

In [4]:
balanced_sections = {}
for key, data in splits.items():
    Xnkm = data["nkm"]
    XG = data['G']
    bins = data['y_bins']
    y = data['y_log']
    print(len(Xnkm),len(XG),len(bins),len(y))
    train_nkm, train_G, y_train_bins, train_y_log = balance_split(Xnkm, XG, bins, y)
    balanced_sections[key] = {
        "nkm": train_nkm,
        "G": np.array(train_G, dtype=np.float32),
        "bins": y_train_bins,
        "y_log": train_y_log
    }



411339 411339 411339 411339
412771 412771 412771 412771
410542 410542 410542 410542
412870 412870 412870 412870
549538 549538 549538 549538
548820 548820 548820 548820
548739 548739 548739 548739
823606 823606 823606 823606
823121 823121 823121 823121
328477 328477 328477 328477
329505 329505 329505 329505
330138 330138 330138 330138
329285 329285 329285 329285
328261 328261 328261 328261
412555 412555 412555 412555
413146 413146 413146 413146
411555 411555 411555 411555
412094 412094 412094 412094
549027 549027 549027 549027
549149 549149 549149 549149
547885 547885 547885 547885


In [5]:
def exp(bin_id, key, noise_std=0.05):
    inputs_nkm = keras.Input(shape=(7,))
    inputs_G = keras.Input(shape=(6,10,))
    #x = layers.Flatten()(inputs_G)
    #x = layers.GaussianNoise(noise_std)(x)
    x = layers.Dense(256)(inputs_G)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(64)(inputs_G)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    # x = layers.Dropout(0.1)(x)
    # x = layers.Dense(16)(inputs_G)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
    # x = layers.BatchNormalization()(x)
    # x = layers.Activation("relu")(x)
    x = layers.Flatten()(x)
    # x = layers.Dropout(0.1)(x)
    # x = layers.Dense(64)(inputs_G)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
    # x = layers.BatchNormalization()(x)
    # x = layers.Activation("relu")(x)
    # x = layers.Flatten()(x)
    shared = layers.Concatenate()([x,inputs_nkm])
    shared = layers.Dense(64, activation='relu')(shared)#kernel_regularizer=tf.keras.regularizers.l2(1e-3),
    out = layers.Dense(1)(x)
    model = keras.Model([inputs_nkm, inputs_G], out, name=f"expert_bin{bin_id}_{key}")
    def logmse(y_true, y_pred):
        sq_diff = tf.square(y_pred - y_true)
        return tf.reduce_mean(sq_diff)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),loss=logmse,metrics=["mae"])
    return model

In [6]:
def logmse(y_true, y_pred):
    sq_diff = tf.square(y_pred - y_true)
    return tf.reduce_mean(sq_diff)
experts={}
for key, data in balanced_sections.items():
    print('\n\n')
    print(key)
    print('\n\n')
    nkm = data["nkm"]
    G = data['G']
    bins = data['bins']
    y = data['y_log']
    expert_dict = OrderedDict()
    for b in sorted(np.unique(bins)):
        ckpt_path = f"expert_{key}_bin{b:02d}.keras"
        if os.path.exists(ckpt_path):
            print(f"Expert {b} checkpoint found, loading {ckpt_path}")
            m = keras.models.load_model(
                ckpt_path,
                compile=False
            )
            m.compile(
                optimizer="adam",
                loss=logmse,
                metrics=["mae"]
            )
            expert_dict[b] = m
            continue
        expert_i = exp(b,key)
        mask = bins == b
        callbacks = [
            keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True),
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
        ]
        
        expert_i.fit([nkm[mask],G[mask]],y[mask],
        validation_split=0.2,
        batch_size=1024, epochs=1000, callbacks = callbacks)
        expert_dict[b] = expert_i

    experts[key] = expert_dict




9_4_2



Expert 0 checkpoint found, loading expert_9_4_2_bin00.keras


I0000 00:00:1745520682.413854   13946 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9571 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6





9_4_3



Expert 0 checkpoint found, loading expert_9_4_3_bin00.keras



9_4_4



Expert 0 checkpoint found, loading expert_9_4_4_bin00.keras



9_4_5



Expert 0 checkpoint found, loading expert_9_4_5_bin00.keras



9_5_2



Expert 0 checkpoint found, loading expert_9_5_2_bin00.keras



9_5_3



Expert 0 checkpoint found, loading expert_9_5_3_bin00.keras



9_5_4



Expert 0 checkpoint found, loading expert_9_5_4_bin00.keras



9_6_2



Expert 0 checkpoint found, loading expert_9_6_2_bin00.keras



9_6_3



Expert 0 checkpoint found, loading expert_9_6_3_bin00.keras



10_4_2



Expert 0 checkpoint found, loading expert_10_4_2_bin00.keras



10_4_3



Expert 0 checkpoint found, loading expert_10_4_3_bin00.keras



10_4_4



Expert 0 checkpoint found, loading expert_10_4_4_bin00.keras



10_4_5



Expert 0 checkpoint found, loading expert_10_4_5_bin00.keras



10_4_6



Expert 0 checkpoint found, loading expert_10_4_6_bin00.keras



10_5_2



Expert 0 checkpoint found, loading expert

In [ ]:
for n, k, m in nkm_list:
    key = f"{n}_{k}_{m}"
    splits[key] = {
        "test_nkm":  joblib.load(f"test_nkm_{n}_{k}_{m}.pkl"),
        "test_G":    joblib.load(f"test_G_{n}_{k}_{m}.pkl"),
        "test_y_log":joblib.load(f"test_ylog_{n}_{k}_{m}.pkl"),
        "test_y_bins":joblib.load(f"test_bins_{n}_{k}_{m}.pkl")
    }

In [ ]:
for cfg, expert_dict in experts.items():
    # grab the test‐split for this config
    data     = splits[cfg]
    Xnkm_te  = np.asarray(data["test_nkm"])
    XG_te    = np.asarray(data["test_G"])
    ylog_te  = np.asarray(data["test_y_log"]).reshape(-1,1)
    bins_te  = np.asarray(data["test_y_bins"])
    
    print(f"\n=== Config {cfg} ({Xnkm_te.shape[0]} samples) ===")
    # for each bin, select the examples and run evaluate()
    for b, model in expert_dict.items():
        mask = (bins_te == b)
        n_b  = mask.sum()
        if n_b == 0:
            print(f"  bin {b:02d}: no test samples")
            continue
        
        Xnkm_b = Xnkm_te[mask]
        XG_b   = XG_te[mask]
        ylog_b = ylog_te[mask]
        
        # evaluate returns [loss, mae] because you compiled with metrics=["mae"]
        loss, mae = model.evaluate(
            [Xnkm_b, XG_b],
            ylog_b,
            verbose=0
        )
        print(f"  bin {b:02d}:  loss={loss:.4f},  mae={mae:.4f},  n={n_b}")

In [ ]:
# nkm_keys = list(experts.keys())
# nkm = np.vstack([ splits[key]["nkm"]  for key in nkm_keys ])
# G   = np.vstack([ splits[key]["G"]    for key in nkm_keys ])
# y_log   = np.vstack([ splits[key]["y_log"] for key in nkm_keys ])


In [ ]:
# for key, data in splits.items():
#     G_tr = data['G']
#     yb_tr = data['y_bins']
#     Bi = len(np.unique(yb_tr))
#     G_in = keras.Input((6,10))
#     x = layers.Dense(128)(G_in)
#     x = layers.BatchNormalization()(x)
#     x = layers.Activation("relu")(x)
#     x = layers.Dropout(0.1)(x)
#     x = layers.Dense(64)(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Activation("relu")(x)
#     #x = layers.Dropout(0.1)(x)
#     x = layers.Flatten()(x)
#     gate = layers.Dense(Bi, activation='softmax')(x)

#     gate_model = keras.Model(G_in, gate, name=f"gate_model_{key}")
#     gate_model.compile(
#         optimizer="adam",
#         loss=tf.keras.losses.SparseCategoricalCrossentropy(),
#         metrics=["accuracy"]
#     )
#     callbacks = [
#             keras.callbacks.ModelCheckpoint(f"gate_{key}.keras", save_best_only=True, monitor="val_accuracy"),
#             keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,restore_best_weights=True),
#             keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1)
#         ]

#     gate_model.fit(
#         G_tr, yb_tr,
#         validation_split=0.2,
#         batch_size=1024,
#         epochs=100,
#         callbacks=callbacks,
#         verbose=2
#     )

#     data["gate_model"] = tf.keras.models.load_model(f"gate_{key}.keras")
#     data["gate_model"].trainable = False
    

In [12]:
config_keys = tf.constant([
  [ 9,4,2],[ 9,4,3],[ 9,4,4],[ 9,4,5],
  [ 9,5,2],[ 9,5,3],[ 9,5,4],
  [ 9,6,2],[ 9,6,3],
  [10,4,2],[10,4,3],[10,4,4],[10,4,5],[10,4,6],
  [10,5,2],[10,5,3],[10,5,4],[10,5,5],
  [10,6,2],[10,6,3],[10,6,4],
], dtype=tf.int32)


class HardGate(layers.Layer):
    def __init__(self, config_keys, **kwargs):
        """
        config_keys: a (21,3) tf.int32 tensor listing your 21 (n,k,m).
        """
        super().__init__(**kwargs)
        self.config_keys = tf.constant(
            config_keys,
            dtype=tf.int32,
            name="config_keys"
        )
        self.n_configs = config_keys.shape[0]

    def call(self, params):
        first3 = tf.cast(params[..., :3], tf.int32)

        expanded = tf.expand_dims(first3, axis=1)
        matches = tf.equal(expanded, self.config_keys[tf.newaxis, ...])
        one_hot = tf.reduce_all(matches, axis=-1)

        return tf.cast(one_hot, tf.float32)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.n_configs)

    def get_config(self):
        base = super().get_config()
        return {**base}

In [ ]:
splits = {}
nkm_list = [
    (9,4,2),(9,4,3),(9,4,4),(9,4,5),
    (9,5,2),(9,5,3),(9,5,4),
    (9,6,2),(9,6,3),
    (10,4,2),(10,4,3),(10,4,4),(10,4,5),(10,4,6),
    (10,5,2),(10,5,3),(10,5,4),(10,5,5),
    (10,6,2),(10,6,3),(10,6,4),
]
for n,k,m in nkm_list:
    key = f"{n}_{k}_{m}"
    nkm = joblib.load(f"train_nkm_{n}_{k}_{m}.pkl")
    G   = joblib.load(f"train_G_{n}_{k}_{m}.pkl")
    y   = joblib.load(f"train_ylog_{n}_{k}_{m}.pkl")
    bins= joblib.load(f"train_bins_{n}_{k}_{m}.pkl")
    splits[key] = {
        "train_nkm":nkm,
        "train_G":G,
        "train_y_bins":bins,
        "train_y_log":y
    }

all_nkm      = []
all_G        = []
all_y_bins   = []
all_y_log    = []

for cfg, data in splits.items():
    all_nkm.append( data["train_nkm"] )
    all_G.append(   data["train_G"] )
    all_y_bins.append(data["train_y_bins"])
    all_y_log.append( data["train_y_log"] )

combined_nkm    = np.vstack(all_nkm)
combined_G      = np.vstack(all_G)
combined_y_bins = np.concatenate(all_y_bins)
combined_y_log  = np.concatenate(all_y_log)

# (Optional) save for next time
joblib.dump(combined_nkm,    "combined_train_nkm.pkl")
joblib.dump(combined_G,      "combined_train_G.pkl")
joblib.dump(combined_y_bins, "combined_train_bins.pkl")
joblib.dump(combined_y_log,  "combined_train_ylog.pkl")

print("All combined:",
      combined_nkm.shape,
      combined_G.shape,
      combined_y_bins.shape,
      combined_y_log.shape)

In [7]:
combined_nkm = joblib.load("combined_train_nkm.pkl")
combined_G = joblib.load("combined_train_G.pkl")
combined_bins = joblib.load("combined_train_bins.pkl")
combined_y_log = joblib.load("combined_train_ylog.pkl")

In [8]:
nkm_tr, nkm_val, G_tr, G_val, y_tr, y_val = train_test_split(
        combined_nkm, combined_G, combined_y_log, test_size=0.2, shuffle=True)
#y_val = 2**y_val
print(max(y_val))

[31.51584]


In [9]:
class Exp2Layer(layers.Layer):
    def call(self, inputs):
        return tf.pow(2.0, inputs)

In [13]:
config_keys = [
  [9,4,2],[9,4,3],[9,4,4],[9,4,5],
  [9,5,2],[9,5,3],[9,5,4],
  [9,6,2],[9,6,3],
  [10,4,2],[10,4,3],[10,4,4],[10,4,5],[10,4,6],
  [10,5,2],[10,5,3],[10,5,4],[10,5,5],
  [10,6,2],[10,6,3],[10,6,4],
]
config_keys = tf.constant(config_keys, dtype=tf.int32)

for m in experts.values():
    print(m[0])
    m[0].trainable = False

inputs_nkm = layers.Input(shape=(7,), name="nkm")
inputs_G   = layers.Input(shape=(6,10), name="G") 

outs = []
for cfg in config_keys.numpy().tolist():
    key = f"{cfg[0]}_{cfg[1]}_{cfg[2]}"
    expert_model = experts[key]
    for expert in expert_model.values():
        out_i = expert([inputs_nkm, inputs_G])
        outs.append(out_i)

all_outputs = layers.Concatenate(axis=1, name="all_configs")(outs)

gate = HardGate(config_keys, name="hard_gate")(inputs_nkm)
h_routed = layers.Dot(axes=1, name="h_final")([gate, all_outputs])
#real_pred = Exp2Layer()(h_routed)

# x = layers.Dense(256)(h_routed)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
# x = layers.BatchNormalization()(x)
# x = layers.Activation("relu")(x)
# x = layers.Dropout(0.1)(x)
# x = layers.Dense(64)(x)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
# x = layers.BatchNormalization()(x)
# x = layers.Activation("relu")(x)
# x = layers.Dropout(0.1)(x)
# x = layers.Dense(16)(x)#, kernel_regularizer=tf.keras.regularizers.l2(1e-3)
# x = layers.BatchNormalization()(x)
# x = layers.Activation("relu")(x)
# x = layers.Dropout(0.1)(x)
# h_final = layers.Dense(1, activation='linear')(h_routed)


router = keras.Model([inputs_nkm, inputs_G], h_routed)
def logmse(y_true, y_pred):
    sq_diff = tf.square(y_pred - y_true)
    return tf.reduce_mean(sq_diff)
def log2_mse(y_true, y_pred):
    y_true = np.log2(y_true)
    y_pred = np.log2(y_pred)
    sq_diff = tf.square(y_pred - y_true)
    return tf.reduce_mean(sq_diff)
router.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),loss=logmse,metrics=["mae"])
# history = expert.fit(
#     [nkm_tr, G_tr], y_tr,
#     validation_data=([nkm_val, G_val], y_val),
#     epochs=100,
#     batch_size=1024,
#     shuffle=True,
#     callbacks=callbacks,
#     verbose=2
# )

<Functional name=expert_bin0_9_4_2, built=True>
<Functional name=expert_bin0_9_4_3, built=True>
<Functional name=expert_bin0_9_4_4, built=True>
<Functional name=expert_bin0_9_4_5, built=True>
<Functional name=expert_bin0_9_5_2, built=True>
<Functional name=expert_bin0_9_5_3, built=True>
<Functional name=expert_bin0_9_5_4, built=True>
<Functional name=expert_bin0_9_6_2, built=True>
<Functional name=expert_bin0_9_6_3, built=True>
<Functional name=expert_bin0_10_4_2, built=True>
<Functional name=expert_bin0_10_4_3, built=True>
<Functional name=expert_bin0_10_4_4, built=True>
<Functional name=expert_bin0_10_4_5, built=True>
<Functional name=expert_bin0_10_4_6, built=True>
<Functional name=expert_bin0_10_5_2, built=True>
<Functional name=expert_bin0_10_5_3, built=True>
<Functional name=expert_bin0_10_5_4, built=True>
<Functional name=expert_bin0_10_5_5, built=True>
<Functional name=expert_bin0_10_6_2, built=True>
<Functional name=expert_bin0_10_6_3, built=True>
<Functional name=expert_bin0_

In [ ]:
splits = {}
nkm_list = [
    (9,4,2),(9,4,3),(9,4,4),(9,4,5),
    (9,5,2),(9,5,3),(9,5,4),
    (9,6,2),(9,6,3),
    (10,4,2),(10,4,3),(10,4,4),(10,4,5),(10,4,6),
    (10,5,2),(10,5,3),(10,5,4),(10,5,5),
    (10,6,2),(10,6,3),(10,6,4),
]
for n,k,m in nkm_list:
    key = f"{n}_{k}_{m}"
    nkm = joblib.load(f"test_nkm_{n}_{k}_{m}.pkl")
    G   = joblib.load(f"test_G_{n}_{k}_{m}.pkl")
    y   = joblib.load(f"test_ylog_{n}_{k}_{m}.pkl")
    bins= joblib.load(f"test_bins_{n}_{k}_{m}.pkl")
    splits[key] = {
        "test_nkm":nkm,
        "test_G":G,
        "test_y_bins":bins,
        "test_y_log":y
    }

all_nkm      = []
all_G        = []
all_y_bins   = []
all_y_log    = []

for cfg, data in splits.items():
    all_nkm.append( data["test_nkm"] )
    all_G.append(   data["test_G"] )
    all_y_bins.append(data["test_y_bins"])
    all_y_log.append( data["test_y_log"] )

combined_nkm    = np.vstack(all_nkm)
combined_G      = np.vstack(all_G)
combined_y_bins = np.concatenate(all_y_bins)
combined_y_log  = np.concatenate(all_y_log)

# (Optional) save for next time
joblib.dump(combined_nkm,    "combined_test_nkm.pkl")
joblib.dump(combined_G,      "combined_test_G.pkl")
joblib.dump(combined_y_bins, "combined_test_bins.pkl")
joblib.dump(combined_y_log,  "combined_test_ylog.pkl")

print("All combined:",
      combined_nkm.shape,
      combined_G.shape,
      combined_y_bins.shape,
      combined_y_log.shape)

In [14]:
combined_nkm = joblib.load("combined_test_nkm.pkl")
combined_G = joblib.load("combined_test_G.pkl")
combined_bins = joblib.load("combined_test_bins.pkl")
combined_y_log = joblib.load("combined_test_ylog.pkl")

In [15]:
#predictions = router.predict([test_nkm, test_G], batch_size=1024)
loss, mae = router.evaluate(
    [combined_nkm, combined_G],  
    combined_y_log,         
    batch_size=1024,
    verbose=2
)

I0000 00:00:1745520720.607751   13997 service.cc:148] XLA service 0x7fecac03a330 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745520720.608093   13997 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3080, Compute Capability 8.6
2025-04-24 13:52:00.648846: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1745520720.798637   13997 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1745520721.240052   13997 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1073/1073 - 5s - 5ms/step - loss: 1.3592 - mae: 0.7670


In [16]:
# for ex in experts.values():
#     ex.trainable = False
# print(len(experts))
# exp_outs = []
# for i in range(20):
#     ex = experts[i]
#     out_i = ex([inputs_nkm, inputs_G])
#     exp_outs.append(out_i)
# experts_tensor = layers.Concatenate(axis=1)(exp_outs)
# h_m_pred = layers.Dot(axes=1, name='h_m_pred')([gate_probs, experts_tensor])

In [ ]:
#take the log of the results

In [ ]:
# from collections import Counter
# train_y_log = np.log2(train_y).reshape(-1, 1)
# val_y_log = np.log2(val_y).reshape(-1, 1)
# test_y_log = np.log2(test_y).reshape(-1, 1)
# print(max(train_y_log), min(train_y_log))
# print(max(val_y_log), min(val_y_log))
# print(max(test_y_log), min(test_y_log))

In [ ]:
# def plot_frequency(data):
#     rounded = [np.round(v) for v in data]
#     unique_vals, counts = np.unique(rounded, return_counts=True)
    
#     plt.figure()
#     plt.bar(unique_vals, counts)
#     plt.xlabel('Value (rounded)')
#     plt.ylabel('Frequency')
#     plt.xticks(unique_vals)

In [ ]:
# plot_frequency(train_y_log)
# plot_frequency(val_y_log)
# plot_frequency(test_y_log)
# plt.show()

In [ ]:
#below organizes the data into 6 different categories that I will use to split the data to be run into my 5 models

In [ ]:
#compile the model

In [ ]:
# def log2_mse(y_true, y_pred):
#     sq_diff = tf.square(y_pred - y_true)
#     return tf.reduce_mean(sq_diff)
# clf_loss = CategoricalCrossentropy(label_smoothing=0.1)
# model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), 
#                loss={"h_m_pred":log2_mse,'gate_probs':clf_loss}, 
#                loss_weights={'h_m_pred':1.0, 'gate_probs':0.2}, 
#                metrics={'h_m_pred':["mae"], 'gate_probs':['accuracy']})
moe.summary()

In [ ]:
#below runs and trains the model

In [ ]:
# history = model1.fit([train_nkm, train_G], 
#                      {"h_m_pred":train_y_log,"gate_probs":y_train_bins}, 
#                      epochs=1000, batch_size = 1024, 
#                      validation_data=([val_nkm, val_G], {"h_m_pred":val_y_log,"gate_probs":y_val_bins}), 
#                      callbacks=callbacks, shuffle=True)

In [ ]:
#Save the model so I do not lose it

In [ ]:
moe.save("dnn_mheight_experts221k_actual_experts.keras") #add graph for loss and mae

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
loss, mae = model1.evaluate([test_nkm, test_G], test_y_log)
print("Test Loss:", loss, "Test MAE:", mae)

In [ ]:
#print a bunch of random points from the test batch of samples

In [ ]:
pred_y_log = model1.predict([test_nkm, test_G])
pred_y = 2 ** pred_y_log

random_indices = np.random.choice(len(test_y), size=100, replace=False)
for i in random_indices:
    n, k, m, _, _,_,_ = test_nkm[i]
    actual = np.log(test_y[i])
    predicted = np.log(pred_y[i])
    print(f"Sample {i+1}: n={n}, k={k}, m={m}")
    print(f"   Actual h_m:    {actual}")
    print(f"   Predicted h_m: {predicted}\n")

In [ ]:
#Function for running the inputs and outputs for the tests

In [ ]:
def recon_G(padded_P, n, k):
    padded_P = padded_P.reshape(6,6)
    P_top = padded_P[:k, :]
    actual_nk = n-k
    P_sub = P_top[:,:actual_nk]

    I = np.eye(k,dtype=padded_P.dtype)

    G = np.concatenate([I, P_sub], axis=1)
    G_pad = np.zeros((6,10), dtype=padded_P.dtype)
    G_pad[:k, :n] = G
    return G_pad

In [ ]:
def transform_P(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['P'],(k,n-k))
    padded = np.zeros((6,6), dtype=G_mat.dtype)
    padded[:G_mat.shape[0], :G_mat.shape[1]] = G_mat

    return padded.flatten()

In [ ]:
def m_height_calculator(inputs, outputs):
    final_nkm = []
    final_P = []
    final_result = []
    for inpt in inputs.keys():
        n, k,m = map(int, inpt.strip("[]").split(','))
        for i in range(len(inputs[inpt])):
            final_nkm.append((n, k, m, m, m, n-k, (n-k)/m))
            sample = {"n": n, "k": k, "m": m, "P": inputs[inpt][i]}
            #print(inputs[inpt][i])
            #print(inputs[inpt][i].shape)
            P = transform_P(sample)
            G = recon_G(P, sample['n'], sample['k']) 
            final_P.append(G)
            final_result.append(np.log2(outputs[inpt][i]))

    final_nkm = np.array(final_nkm, dtype=np.float32)
    final_P = np.array(final_P, dtype=np.float32)
    
    model1 = load_model("dnn_mheight_experts221k_P3_1.keras",custom_objects={"log2_mse": log2_mse})
    pred_y_log = model1.predict([final_nkm, final_P])
    pred_y = 2 ** pred_y_log

    return pred_y_log, final_result
        
            

In [ ]:
#Add inputs and outputs below and run them

In [ ]:
inputs={
        '[9,4,2]': [
            np.array([[ 12.34182835,  78.8825531,  -74.04528809, -93.71873474,  76.5475769 ],
             [ 70.00410461,  31.84829903,  10.89520359, -13.96429634,  51.48582458],
             [-34.13206482,  38.65733337, -17.05139732,  87.81211853, -51.16646194],
             [-52.76997375,  -4.83906269, -59.08993912, -43.87838745,  48.50744629]]
            ),
            np.array([[-20.93441391,  16.07522583, -74.35280609, -92.421875,    31.05505753],
 [-67.70335388, -10.42372894,  71.59892273, -92.68517303,  33.64113998],
 [ 81.48065948, -49.71315765, -43.87363052,  75.6060791,   35.69181061],
 [  6.75473022, -63.70702362, -19.96879578,  17.10869789,  11.18667507]]),
        ],
    }
outputs={
        '[9,4,2]': [
            354.5741951135599,
            703.001648767227
        ]
    }
pred, result = m_height_calculator(inputs, outputs)
# print(len(pred))
# G = G_generator(9,4)
# sample = {"n": 9, "k": 4, "m": 2, "P": G}
# h = H_m_computer(G, 9, 4)
# print(G)
# P = G[:4,4:]
# print(P)
# print(h)
# G = G_generator(9,4)
# sample = {"n": 9, "k": 4, "m": 2, "P": G}
# h = H_m_computer(G, 9, 4)
# P = G[:4,4:]
# print(P)
# print(h)

In [ ]:
print(result, pred)